# 🚀 Factorized Context MIL - Azure ML Job Submit
---
이 노트북은 `factorized_context_mil.py` 및 `train_fc_mil.py`를 사용하여 Factorized Context MIL 모델 학습 작업을 Azure ML에 제출합니다.

## 📦 패키지 Import

In [1]:
from pathlib import Path
from azure.ai.ml import MLClient, command, Input
from azure.ai.ml.entities import Environment
from azure.identity import DefaultAzureCredential, InteractiveBrowserCredential
from azure.ai.ml.constants import AssetTypes
import json
from datetime import datetime

print("✅ 패키지 import 완료")

✅ 패키지 import 완료


## 🎛️ 하이퍼파라미터 설정

In [2]:
# ============================================================================
# 🎯 실험 설정
# ============================================================================
EXPERIMENT_NAME = "CAFA6_FC_MIL_Training"
DISPLAY_NAME = f"fc_mil_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

# ============================================================================
# 🖥️ 컴퓨팅 설정
# ============================================================================
CLUSTER_NAME = "FOR-CAFA-6"  # 클러스터 이름
INSTANCE_COUNT = 1

# ============================================================================
# 📊 데이터셋 설정
# ============================================================================
DATASET_NAME = "cafa_6"
DATASET_VERSION = "6"
DATASET_FOLDER_NAME = "CAFA6_82404sq_Asymmetric_Loss"

# ============================================================================
# 🐍 환경 설정
# ============================================================================
ENVIRONMENT_NAME = "cafa6-torch-env"
USE_EXISTING_ENV = True

# ============================================================================
# 🏗️ 모델 아키텍처 (Factorized Context MIL)
# ============================================================================
EMBEDDING_DIM = 2560
NUM_CLASSES = 40122
MIL_HIDDEN_DIM = 512
DROPOUT = 0.3

# [New] Factorized Context Params
NUM_LATENT_FACTORS = 64
TEMPERATURE = 10.0
SEMANTIC_EMBEDDING_FILENAME = "embedded_go_by_biobert/go_embeddings_optimized.npy"

# ============================================================================
# 🏋️ 학습 설정
# ============================================================================
BATCH_SIZE = 64
NUM_EPOCHS = 50
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 1e-4
GRAD_CLIP_NORM = 1.0

# ============================================================================
# 📉 스케줄러 설정
# ============================================================================
SCHEDULER_TYPE = 'cosine_warm_restarts'
SCHEDULER_FACTOR = 0.5
SCHEDULER_PATIENCE = 3
SCHEDULER_MIN_LR = 1e-6
COSINE_T_0 = 13
COSINE_T_MULT = 2
COSINE_RESTART_DECAY = 0.8

# ============================================================================
# ⏸️ Early Stopping
# ============================================================================
EARLY_STOP_PATIENCE = 10
EARLY_STOP_MIN_DELTA = 1e-4

# ============================================================================
# ⚡ 최적화
# ============================================================================
USE_AMP = True
NUM_WORKERS = 8
VAL_SPLIT = 0.1
RANDOM_SEED = 42

# ============================================================================
# 🔥 손실 함수 설정
# ============================================================================
USE_IC_WEIGHTED_LOSS = True
GAMMA_POS = 0.0
GAMMA_NEG = 4.0
ASL_CLIP = 0.05

print("✅ 설정 완료")
print(f"   Latent Factors: {NUM_LATENT_FACTORS}")
print(f"   Temperature: {TEMPERATURE}")
print(f"   Semantic Embeddings: {SEMANTIC_EMBEDDING_FILENAME}")

✅ 설정 완료
   Latent Factors: 64
   Temperature: 10.0
   Semantic Embeddings: embedded_go_by_biobert/go_embeddings_optimized.npy


## 🔐 Azure ML 연결

In [3]:
print("=" * 70)
print("🔐 Azure ML Client 연결 시작")
print("=" * 70)

try:
    try:
        credential = DefaultAzureCredential()
        credential.get_token("https://management.azure.com/.default")
    except Exception:
        print("🌐 브라우저 인증(Interactive)으로 전환합니다...")
        credential = InteractiveBrowserCredential()

    ml_client = MLClient.from_config(credential=credential)
    ws = ml_client.workspaces.get(ml_client.workspace_name)

    print(f"✅ 연결 성공!")
    print(f"   - Workspace: {ws.name}")
    print(f"   - Resource Group: {ws.resource_group}")
    print(f"   - Subscription: {ml_client.subscription_id}")

except Exception as e:
    print(f"❌ 연결 실패: {e}")

🔐 Azure ML Client 연결 시작


InteractiveBrowserBrokerCredential.get_token failed: (pii). Status: Response_Status.Status_InteractionRequired, Error code: 3399548929, Tag: 527291998
InteractiveBrowserBrokerCredential.get_token failed: (pii). Status: Response_Status.Status_InteractionRequired, Error code: 3399548929, Tag: 527291998
DefaultAzureCredential failed to retrieve a token from the included credentials.
Attempted credentials:
	EnvironmentCredential: EnvironmentCredential authentication unavailable. Environment variables are not fully configured.
Visit https://aka.ms/azsdk/python/identity/environmentcredential/troubleshoot to troubleshoot this issue.
	WorkloadIdentityCredential: WorkloadIdentityCredential authentication unavailable. The workload options are not fully configured. See the troubleshooting guide for more information: https://aka.ms/azsdk/python/identity/workloadidentitycredential/troubleshoot. Missing required arguments: 'tenant_id', 'client_id', 'token_file_path'.
	ManagedIdentityCredential: Mana

🌐 브라우저 인증(Interactive)으로 전환합니다...


Class DeploymentTemplateOperations: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.


✅ 연결 성공!
   - Workspace: REDACTED_AZURE_RESOURCE_GROUP-ml
   - Resource Group: REDACTED_AZURE_RESOURCE_GROUP
   - Subscription: REDACTED_AZURE_SUBSCRIPTION_ID


## 🐍 환경 설정

In [4]:
env_name = ENVIRONMENT_NAME

if USE_EXISTING_ENV:
    try:
        env = ml_client.environments.get(name=env_name, label="latest")
        print(f"✅ 기존 환경 사용: {env_name} (version: {env.version})")
    except Exception as e:
        print(f"⚠️ 기존 환경 찾을 수 없음: {e}")
        USE_EXISTING_ENV = False

if not USE_EXISTING_ENV:
    conda_yaml_path = Path("env/conda.yaml")
    if not conda_yaml_path.exists():
        raise FileNotFoundError(f"env/conda.yaml 파일을 찾을 수 없습니다.")
    
    env = Environment(
        name=env_name,
        description="CAFA 6 - PyTorch 환경",
        conda_file=str(conda_yaml_path),
        image="mcr.microsoft.com/azureml/openmpi4.1.0-cuda11.8-cudnn8-ubuntu22.04:latest",
    )
    env = ml_client.environments.create_or_update(env)
    print(f"✅ 환경 생성 완료: {env_name} (version: {env.version})")

✅ 기존 환경 사용: cafa6-torch-env (version: 1)


## 🚀 Job 생성 및 제출

In [5]:
dataset_input = Input(
    type=AssetTypes.URI_FOLDER,
    path=f"azureml:{DATASET_NAME}:{DATASET_VERSION}"
)

src_folder = Path("src")

# Command 생성
command_str = f"""
python train_fc_mil.py \
  --data_path ${{{{inputs.dataset}}}} \
  --dataset_name '{DATASET_FOLDER_NAME}' \
  --embedding_dim {EMBEDDING_DIM} \
  --num_classes {NUM_CLASSES} \
  --mil_hidden_dim {MIL_HIDDEN_DIM} \
  --dropout {DROPOUT} \
  --num_latent_factors {NUM_LATENT_FACTORS} \
  --temperature {TEMPERATURE} \
  --semantic_embedding_filename '{SEMANTIC_EMBEDDING_FILENAME}' \
  --batch_size {BATCH_SIZE} \
  --num_epochs {NUM_EPOCHS} \
  --learning_rate {LEARNING_RATE} \
  --weight_decay {WEIGHT_DECAY} \
  --grad_clip_norm {GRAD_CLIP_NORM} \
  {'--use_ic_weighted_loss' if USE_IC_WEIGHTED_LOSS else ''} \
  --gamma_pos {GAMMA_POS} \
  --gamma_neg {GAMMA_NEG} \
  --asl_clip {ASL_CLIP} \
  --scheduler_type {SCHEDULER_TYPE} \
  --scheduler_factor {SCHEDULER_FACTOR} \
  --scheduler_patience {SCHEDULER_PATIENCE} \
  --scheduler_min_lr {SCHEDULER_MIN_LR} \
  --cosine_t_0 {COSINE_T_0} \
  --cosine_t_mult {COSINE_T_MULT} \
  --cosine_restart_decay {COSINE_RESTART_DECAY} \
  --early_stop_patience {EARLY_STOP_PATIENCE} \
  --early_stop_min_delta {EARLY_STOP_MIN_DELTA} \
  --num_workers {NUM_WORKERS} \
  --val_split {VAL_SPLIT} \
  --random_seed {RANDOM_SEED} \
  {'--use_amp' if USE_AMP else ''} \
  --output_dir ./outputs
""".strip()

job = command(
    code=str(src_folder),
    command=command_str,
    inputs={"dataset": dataset_input},
    environment=f"{env.name}:{env.version}",
    compute=CLUSTER_NAME,
    instance_count=INSTANCE_COUNT,
    experiment_name=EXPERIMENT_NAME,
    display_name=DISPLAY_NAME,
    description=f"FC-MIL (Latents={NUM_LATENT_FACTORS}, Temp={TEMPERATURE}, Hidden={MIL_HIDDEN_DIM})",
)

print("📤 Job 제출 중...")
try:
    returned_job = ml_client.jobs.create_or_update(job)
    print(f"✅ Job 제출 완료: {returned_job.name}")
    print(f"🔗 URL: {returned_job.studio_url}")
except Exception as e:
    print(f"❌ Job 제출 실패: {e}")

📤 Job 제출 중...


Class AutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class AutoDeleteConditionSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseAutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class IntellectualPropertySchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class ProtectionLevelSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseIntellectualPropertySchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Uploading src (0.5 MBs): 100%|########

✅ Job 제출 완료: bright_yuca_mc0g9cg3jt
🔗 URL: https://ml.azure.com/runs/bright_yuca_mc0g9cg3jt?wsid=/subscriptions/REDACTED_AZURE_SUBSCRIPTION_ID/resourcegroups/REDACTED_AZURE_RESOURCE_GROUP/workspaces/REDACTED_AZURE_RESOURCE_GROUP-ml&tid=5fb256f0-fbf2-40d2-81d5-bac1b32c419d
